# 向量检索（Vector Search）

**用途**：教学演示 - 基于向量相似度的语义检索

**核心概念**：
- Embedding（文本向量化）
- 相似度计算（余弦相似度、欧氏距离）
- 近似最近邻搜索（ANN）

## 运行前检查

1. 已安装依赖：pip install -r ../requirements.txt
2. 已完成 01_scalar_query.py 的学习
3. 确保 Milvus 服务可用（会自动创建测试数据）

In [3]:
from rag_examples.milvus_config import MILVUS_URI
from dotenv import load_dotenv
load_dotenv()

True

## 理解向量检索

### 定义
向量检索 = 基于向量相似度的检索
将文本转换为向量后，在向量空间中查找最相似的文档

### 生活化比喻
向量检索 = "按感觉找东西"

传统搜索（关键字匹配）：
- 找"苹果"→ 找包含"苹果"这两个字的文档
- 找不到"iPhone"、"水果"相关内容

向量检索（语义匹配）：
- 找"苹果"→ 找和"苹果"语义接近的文档
- 能找到"iPhone"、"水果"、"MacBook"相关内容

### 核心流程
1. 文本 → Embedding 模型 → 向量
2. 查询向量与库中向量逐一比较相似度
3. 返回最相似的 Top-K 文档

### 相似度度量
1. **余弦相似度（Cosine Similarity）**
   - 计算两个向量夹角的余弦值
   - 范围：[-1, 1]，越接近 1 越相似

2. **欧氏距离（L2 Distance）**
   - 计算两个向量在空间中的直线距离
   - 范围：[0, +∞)，越小越相似

3. **内积（Inner Product）**
   - 两个向量对应位置相乘再求和
   - 范围：(-∞, +∞)，越大越相似

### 为什么需要向量检索？
1. 语义匹配：理解"什么意思"，不只是"有什么词"
2. 跨语言：中文"苹果"能匹配英文"apple"
3. 容错性：打错字也能找到相关内容
4. 多模态：文本、图像、音频都可以转为向量

## 示例 1: 准备测试数据

In [4]:
def prepare_test_collection():
    from pymilvus import MilvusClient
    import random

    random.seed(42)

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "vector_search_demo"

    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE"
    )

    documents = [
        {"content": "人工智能是模拟人类智能的计算机科学领域，包括机器学习、深度学习等技术。", "category": "AI"},
        {"content": "机器学习通过训练数据让计算机自动学习规律，无需显式编程。", "category": "AI"},
        {"content": "深度学习使用多层神经网络，在图像识别和自然语言处理领域取得成功。", "category": "AI"},
        {"content": "产品设计需要考虑用户体验、功能完整性和技术可行性。", "category": "Product"},
        {"content": "好的产品设计应该简洁易用，解决用户的实际问题。", "category": "Product"},
        {"content": "Python 是一门流行的编程语言，适合数据科学和人工智能开发。", "category": "Programming"},
        {"content": "编程是将人类思维转化为计算机可执行指令的过程。", "category": "Programming"},
    ]

    def semantic_embedding(category, index):
        random.seed(category + str(index))
        base = [0.5 if i % 3 == 0 else -0.3 for i in range(768)]
        if category == "AI":
            base[0:100] = [0.8] * 100
        elif category == "Product":
            base[100:200] = [0.8] * 100
        elif category == "Programming":
            base[200:300] = [0.8] * 100
        vector = [b + random.uniform(-0.1, 0.1) for b in base]
        return vector

    data_to_insert = []
    for i, doc in enumerate(documents):
        data_to_insert.append({
            "content": doc["content"],
            "category": doc["category"],
            "vector": semantic_embedding(doc["category"], i)
        })

    client.insert(collection_name, data_to_insert)

    print(f"✓ 已创建测试集合：{collection_name}")
    print(f"  插入文档数：{len(documents)}")
    print(f"  文档类别：AI(3 条), Product(2 条), Programming(2 条)")

    return client, collection_name

In [5]:
client, collection_name = prepare_test_collection()


✓ 已创建测试集合：vector_search_demo
  插入文档数：7
  文档类别：AI(3 条), Product(2 条), Programming(2 条)


## 示例 2: 基础向量检索

In [6]:
def basic_vector_search(client, collection_name):
    print("=" * 60)
    print("示例 2: 基础向量检索")
    print("=" * 60)

    import random
    random.seed(100)

    print("\n场景 1: 查询'机器学习是什么'")
    print("  构造 AI 相关的查询向量...")
    query_vector = [0.8 if i < 100 else random.uniform(-0.2, 0.2) for i in range(768)]

    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=3,
        output_fields=["content", "category"]
    )

    print("\n检索结果（Top-3）：")
    for i, hit in enumerate(results[0]):
        print(f"  [{i+1}] 相似度：{hit['distance']:.4f}")
        print(f"      类别：{hit['entity']['category']}")
        print(f"      内容：{hit['entity']['content'][:50]}...")
        print()

    print("场景 2: 查询'如何设计产品'")
    print("  构造 Product 相关的查询向量...")
    query_vector = [0.8 if 100 <= i < 200 else random.uniform(-0.2, 0.2) for i in range(768)]

    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=3,
        output_fields=["content", "category"]
    )

    print("\n检索结果（Top-3）：")
    for i, hit in enumerate(results[0]):
        print(f"  [{i+1}] 相似度：{hit['distance']:.4f}")
        print(f"      类别：{hit['entity']['category']}")
        print(f"      内容：{hit['entity']['content'][:50]}...")
        print()

In [7]:
basic_vector_search(client, collection_name)

示例 2: 基础向量检索

场景 1: 查询'机器学习是什么'
  构造 AI 相关的查询向量...

检索结果（Top-3）：
  [1] 相似度：0.5783
      类别：AI
      内容：深度学习使用多层神经网络，在图像识别和自然语言处理领域取得成功。...

  [2] 相似度：0.5755
      类别：AI
      内容：机器学习通过训练数据让计算机自动学习规律，无需显式编程。...

  [3] 相似度：0.5755
      类别：AI
      内容：人工智能是模拟人类智能的计算机科学领域，包括机器学习、深度学习等技术。...

场景 2: 查询'如何设计产品'
  构造 Product 相关的查询向量...

检索结果（Top-3）：
  [1] 相似度：0.5969
      类别：Product
      内容：好的产品设计应该简洁易用，解决用户的实际问题。...

  [2] 相似度：0.5959
      类别：Product
      内容：产品设计需要考虑用户体验、功能完整性和技术可行性。...

  [3] 相似度：-0.0147
      类别：AI
      内容：人工智能是模拟人类智能的计算机科学领域，包括机器学习、深度学习等技术。...



## 示例 3: 不同度量类型的对比

In [ ]:
def metric_type_comparison():
    print("=" * 60)
    print("示例 3: 不同度量类型对比")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 度量类型对比                                            │
├──────────────┬────────────────┬─────────────────────────┤
│    类型       │    计算方式     │         特点             │
├──────────────┼────────────────┼─────────────────────────┤
│ COSINE       │ 余弦相似度      │ 值越大越相似 (范围 -1~1) │
│              │ cos(θ)         │ 不受向量长度影响         │
│              │                │ 推荐作为默认选择         │
├──────────────┼────────────────┼─────────────────────────┤
│ L2           │ 欧几里得距离    │ 值越小越相似             │
│ (EUCLIDEAN)  │ √(Σ(xi-yi)²)   │ 考虑向量绝对位置         │
│              │                │ 适用于未归一化的向量     │
├──────────────┼────────────────┼─────────────────────────┤
│ IP           │ 内积            │ 值越大越相似             │
│ (INNER_PRODUCT)│ Σ(xi*yi)     │ 归一化后与 COSINE 等价   │
│              │                │ 某些场景计算更快         │
└──────────────┴────────────────┴─────────────────────────┘

如何选择？

1. COSINE（推荐）
   ✓ 适用于大多数语义检索场景
   ✓ Embedding 模型通常输出归一化向量
   ✓ 结果直观（-1 到 1 的相似度）

2. L2
   ✓ 向量未归一化时使用
   ✓ 需要考虑向量绝对大小的场景

3. IP
   ✓ 与 COSINE 在归一化后等价
   ✓ 追求极致性能时使用
""")

metric_type_comparison()

## 示例 4: 批量向量检索

In [ ]:
def batch_vector_search(client, collection_name):
    print()
    print("=" * 60)
    print("示例 4: 批量向量检索")
    print("=" * 60)

    import random

    query_vectors = [
        [0.8 if i < 100 else random.uniform(-0.2, 0.2) for i in range(768)],
        [0.8 if 100 <= i < 200 else random.uniform(-0.2, 0.2) for i in range(768)],
        [0.8 if 200 <= i < 300 else random.uniform(-0.2, 0.2) for i in range(768)],
    ]

    print("批量查询 3 个问题向量...\n")

    results = client.search(
        collection_name=collection_name,
        data=query_vectors,
        limit=2,
        output_fields=["category"]
    )

    for i, result in enumerate(results):
        category = ["AI", "产品", "编程"][i]
        print(f"【查询向量 {i+1}】({category}相关)")
        for j, hit in enumerate(result):
            print(f"  [{j+1}] 相似度：{hit['distance']:.4f} | 类别：{hit['entity']['category']}")
        print()

In [ ]:
batch_vector_search(client, collection_name)

## 示例 5: 向量检索参数详解

In [ ]:
def search_params_explained(client, collection_name):
    print("=" * 60)
    print("示例 5: 向量检索参数详解")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 向量检索参数说明                                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. limit (Top-K)                                        │
│    含义：返回最相似的 K 个结果                            │
│    建议：5-20（太多会增加后续处理负担）                 │
│                                                         │
│ 2. output_fields                                        │
│    含义：返回哪些字段的内容                             │
│    建议：只返回需要的字段，减少数据传输                 │
│                                                         │
│ 3. filter（标量过滤）                                   │
│    含义：在向量检索基础上增加条件筛选                   │
│    示例："category == 'AI' and views > 1000"            │
│                                                         │
│ 4. search_params（索引相关参数）                        │
│    IVF_FLAT 索引：{"nprobe": 10}                        │
│    HNSW 索引：{"ef": 64}                                │
│                                                         │
└─────────────────────────────────────────────────────────┘
""")

    import random
    query_vector = [random.random() for _ in range(768)]

    print("实际检索演示：")
    print("-" * 50)

    results = client.search(
        collection_name=collection_name,
        data=[query_vector, query_vector],
        limit=5,
        output_fields=["content", "category"],
        search_params={"nprobe": 10}
    )

    print(f"返回 {len(results[0])} 个结果：")
    for i, hit in enumerate(results[0]):
        print(f"  [{i+1}] 相似度：{hit['distance']:.4f} | 类别：{hit['entity']['category']}")

In [ ]:
search_params_explained(client, collection_name)

## 示例 6: 使用真实 Embedding 模型

In [ ]:
def search_with_real_embedding():
    print()
    print("=" * 60)
    print("示例 6: 使用真实 Embedding 模型")
    print("=" * 60)

    print("""
实际项目中如何生成查询向量？

方法 1: sentence-transformers（本地模型）
────────────────────────────────────────
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
query = "什么是机器学习"
query_vector = model.encode(query).tolist()

方法 2: OpenAI Embedding API
────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key="your-key")
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="什么是机器学习"
)
query_vector = response.data[0].embedding

方法 3: 阿里云百炼 API
────────────────────────────────────────
from dashscope import TextEmbedding

result = TextEmbedding.call(
    model='text-embedding-v2',
    input='什么是机器学习'
)
query_vector = result.output['embeddings'][0]['embedding']
""")

    print("模拟流程：")
    print("-" * 50)

    queries = [
        "人工智能和机器学习有什么关系？",
        "如何设计一个好的产品？",
        "Python 适合做什么开发？"
    ]

    for query in queries:
        print(f"  用户问题：{query}")
        print(f"  ↓ Embedding 模型")
        print(f"  输出：768 维向量 [0.023, -0.045, 0.089, ...]")
        print(f"  ↓ 向量检索")
        print(f"  返回：Top-3 最相似的文档")
        print()

In [ ]:
search_with_real_embedding()

## 示例 7: 向量检索最佳实践

In [ ]:
def vector_search_best_practices():
    print()
    print("=" * 60)
    print("示例 7: 向量检索最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 向量检索最佳实践                                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. Embedding 模型选择                                    │
│    - 中文场景：bge-large-zh, m3e-base                   │
│    - 多语言：paraphrase-multilingual-MiniLM             │
│    - 通用：text-embedding-3-small (OpenAI)              │
│                                                         │
│ 2. 向量维度匹配                                          │
│    - 确保查询向量维度与 Collection 定义一致              │
│    - 常见维度：768 (bert), 1536 (OpenAI), 1024          │
│                                                         │
│ 3. 相似度阈值                                            │
│    - 建议设置最低相似度阈值过滤噪声                     │
│    - COSINE 场景：threshold=0.5-0.7                      │
│                                                         │
│ 4. 结果数量选择                                          │
│    - Top-K 建议：5                                   │
│    - 太多：增加 LLM 上下文负担                          │
│    - 太少：可能遗漏相关信息                             │
│                                                         │
│ 5. 性能优化                                              │
│    - 数据量>1 万时建立索引（IVF_FLAT/HNSW）             │
│    - 大批量查询时分批处理                               │
│                                                         │
└─────────────────────────────────────────────────────────┘

RAG 系统中的典型用法:

完整的 RAG 检索流程：

```python
def retrieve_context(query, top_k=5):
    query_vector = embedding_model.encode(query).tolist()
    results = client.search(
        collection_name="knowledge_base",
        data=[query_vector],
        limit=top_k,
        output_fields=["content", "title"]
    )
    contexts = []
    for hit in results[0]:
        if hit['distance'] > 0.5:
            contexts.append(hit['entity']['content'])
    return contexts
```
""")

vector_search_best_practices()

## 总结

向量检索是 RAG 系统的核心：
- 将文本转为向量进行语义匹配
- 支持跨语言、容错性搜索
- 结合标量过滤实现精准检索

下一步：03_keyword_search.py（关键字检索/BM25）